### asyncio vs anyio

In [ ]:
import asyncio
import time

import anyio

start_time = time.monotonic()

def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")

**Task groups — asyncio**

In [ ]:
async def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    try:
        await asyncio.sleep(brew)
    except asyncio.CancelledError:
        log(f"{item}: thrown out")
        raise
    if fail:
        raise RuntimeError(f"out of {item}")
    return item


async def run_task_group_asyncio():
    async with asyncio.TaskGroup() as group:
        task_a = group.create_task(make_item("espresso", brew=0.5))
        task_b = group.create_task(make_item("latte", brew=0.5))
    return [task_a.result(), task_b.result()]


results = await run_task_group_asyncio()
print(results)

**Task groups — anyio** (same `make_item`, on `anyio.sleep`)

In [ ]:
async def make_item_anyio(item: str = "coffee", brew: float = 2.0) -> str:
    await anyio.sleep(brew)
    return item


async def run_task_group_anyio():
    results = []

    async with anyio.create_task_group() as group:
        async def collect(item):
            results.append(await make_item_anyio(item, brew=0.5))

        group.start_soon(collect, "espresso")
        group.start_soon(collect, "latte")
    return results


results = await run_task_group_anyio()
print(results)

**Timeouts — asyncio**

In [ ]:
try:
    async with asyncio.timeout(1.5):
        result = await make_item("slow drip", brew=5.0)
except TimeoutError:
    log("asyncio.timeout: timed out")

**Timeouts — anyio**

In [ ]:
try:
    with anyio.fail_after(1.5):
        result = await make_item_anyio("slow drip", brew=5.0)
except TimeoutError:
    log("anyio.fail_after: timed out")

**`move_on_after` — non-raising**

In [ ]:
start_time = time.monotonic()

with anyio.move_on_after(1.5) as scope:
    await make_item_anyio("slow drip", brew=5.0)

log(f"cancel_called={scope.cancel_called}")

**Concurrency limit — asyncio**

In [ ]:
async def serve_with_limit(order: str, rate_limiter: asyncio.Semaphore):
    async with rate_limiter:
        log(f"serving {order}")
        await make_item(order, brew=1.0)
        return f"{order}: done"


rate_limiter = asyncio.Semaphore(3)

results = await asyncio.gather(*(serve_with_limit(f"order-{index}", rate_limiter) for index in range(6)))
for entry in results:
    log(entry)

**Concurrency limit — anyio**

In [ ]:
async def serve_with_limit_anyio(order: str, capacity: anyio.CapacityLimiter):
    async with capacity:
        log(f"serving {order}")
        await make_item_anyio(order, brew=1.0)
        return f"{order}: done"


capacity = anyio.CapacityLimiter(3)

collected: list = []
async with anyio.create_task_group() as group:
    async def run(order):
        collected.append(await serve_with_limit_anyio(order, capacity))

    for index in range(6):
        group.start_soon(run, f"order-{index}")

for entry in collected:
    log(entry)

**Events — asyncio (reusable via `clear()`)**

In [ ]:
signal = asyncio.Event()


async def waiter_asyncio(name: str):
    log(f"{name}: waiting")
    await signal.wait()
    log(f"{name}: released")


waiters = [asyncio.create_task(waiter_asyncio(f"w{index}")) for index in range(3)]
await asyncio.sleep(0.5)
signal.set()
await asyncio.gather(*waiters)

log(f"before clear: is_set()={signal.is_set()}")
signal.clear()
log(f"after clear:  is_set()={signal.is_set()} — ready to reuse")

**Events — anyio (single-use, no `clear()`)**

In [ ]:
signal = anyio.Event()


async def waiter_anyio(name: str):
    log(f"{name}: waiting")
    await signal.wait()
    log(f"{name}: released")


async with anyio.create_task_group() as group:
    group.start_soon(waiter_anyio, "w0")
    group.start_soon(waiter_anyio, "w1")
    group.start_soon(waiter_anyio, "w2")
    await anyio.sleep(0.5)
    signal.set()

log(f"is_set()={signal.is_set()}")
try:
    signal.clear()
    log("clear() worked (unexpected)")
except AttributeError as error:
    log(f"anyio.Event has no clear(): {error}")

---
| feature | asyncio | anyio |
|---|---|---|
| task group | `asyncio.TaskGroup()` | `anyio.create_task_group()` + `start_soon` |
| timeout (raising) | `asyncio.timeout(t)` | `anyio.fail_after(t)` |
| timeout (silent) | — | `anyio.move_on_after(t)` |
| concurrency limit | `Semaphore(n)` | `CapacityLimiter(n)` |
| event | reusable (`clear()`) | single-use |